In [ ]:
# Bibliotecas usadas no projeto
import numpy as np   # cálculos e condições
import pandas as pd  # tabelas de dados

In [ ]:
# Conecta o Colab ao Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

# Pasta do Drive onde estão os arquivos CSV
PASTA = "/content/drive/MyDrive/projetoCinema/"

# Lista os arquivos da pasta para conferir
print(os.listdir(PASTA))

['Dim_Filme.csv', 'Dim_Sala.csv', 'Fato_Sessao.csv', 'Fato_Venda_Bomboniere.csv', 'Fato_Venda_Ingresso.csv', 'dados_tratados']


In [ ]:
# Lê cada arquivo CSV e guarda em uma tabela (DataFrame)
filmes = pd.read_csv(PASTA + "Dim_Filme.csv")
salas = pd.read_csv(PASTA + "Dim_Sala.csv")
sessoes = pd.read_csv(PASTA + "Fato_Sessao.csv")
ingressos = pd.read_csv(PASTA + "Fato_Venda_Ingresso.csv")
bomboniere = pd.read_csv(PASTA + "Fato_Venda_Bomboniere.csv")

In [ ]:
# shape devolve (número de linhas, número de colunas)
print("filmes:", filmes.shape)
print("salas:", salas.shape)
print("sessoes:", sessoes.shape)
print("ingressos:", ingressos.shape)
print("bomboniere:", bomboniere.shape)

filmes: (20, 5)
salas: (20, 4)
sessoes: (336, 5)
ingressos: (22235, 6)
bomboniere: (7454, 5)


In [ ]:
# head() mostra as 5 primeiras linhas da tabela
filmes.head()

,ID_Filme,Titulo,Genero,Classificacao_Indicativa,Data_Lancamento
0,1,O Vingador do Futuro,Ação,16 Anos,2023-11-01
1,2,Estrelas Além do Tempo,Drama,Livre,2023-10-15
2,3,Riso Solto,Comédia,12 Anos,2023-11-10
3,4,O Código Oculto,Ficção Científica,14 Anos,2023-09-20
4,5,Aventuras na Selva,Aventura,Livre,2023-10-05


In [ ]:
## 2. Diagnóstico da qualidade dos dados

In [ ]:
# Dicionário com as 5 tabelas, para percorrer todas de uma vez
tabelas = {
    "Dim_Filme": filmes,
    "Dim_Sala": salas,
    "Fato_Sessao": sessoes,
    "Fato_Venda_Ingresso": ingressos,
    "Fato_Venda_Bomboniere": bomboniere,
}

In [ ]:
# Conta valores nulos e linhas duplicadas em cada tabela
for nome, df in tabelas.items():
    nulos = df.isna().sum().sum()
    duplicadas = df.duplicated().sum()
    print(nome, "| nulos:", nulos, "| linhas duplicadas:", duplicadas)

Dim_Filme | nulos: 0 | linhas duplicadas: 0
Dim_Sala | nulos: 0 | linhas duplicadas: 0
Fato_Sessao | nulos: 0 | linhas duplicadas: 0
Fato_Venda_Ingresso | nulos: 0 | linhas duplicadas: 0
Fato_Venda_Bomboniere | nulos: 0 | linhas duplicadas: 0


In [ ]:
# Mostra o tipo de dado de cada coluna
for nome, df in tabelas.items():
    print("---", nome)
    print(df.dtypes)
    print()

--- Dim_Filme
ID_Filme                     int64
Titulo                      object
Genero                      object
Classificacao_Indicativa    object
Data_Lancamento             object
dtype: object

--- Dim_Sala
ID_Sala        int64
Nome_Sala     object
Tipo_Sala     object
Capacidade     int64
dtype: object

--- Fato_Sessao
ID_Sessao            int64
ID_Filme             int64
ID_Sala              int64
Data_Hora_Sessao    object
Preco_Base           int64
dtype: object

--- Fato_Venda_Ingresso
ID_Venda_Ingresso      int64
ID_Sessao              int64
Tipo_Ingresso         object
Canal_Venda           object
Data_Venda            object
Valor_Pago           float64
dtype: object

--- Fato_Venda_Bomboniere
ID_Venda_Bomboniere     int64
ID_Sessao               int64
Item_Comprado          object
Quantidade              int64
Valor_Total             int64
dtype: object



In [ ]:
# Valores distintos das colunas de categoria
print(filmes["Genero"].unique())
print(salas["Tipo_Sala"].unique())
print(ingressos["Tipo_Ingresso"].unique())
print(ingressos["Canal_Venda"].unique())

['Ação' 'Drama' 'Comédia' 'Ficção Científica' 'Aventura' 'Terror'
 'Romance' 'Suspense']
['Padrão' '3D XPLUS' 'VIP' 'IMAX']
['Meia' 'Inteira']
['Site' 'Aplicativo' 'Totem' 'Bilheteria']


In [ ]:
## 5. Tratamento
# 5.1 Texto -> data e hora
filmes["Data_Lancamento"] = pd.to_datetime(filmes["Data_Lancamento"])
sessoes["Data_Hora_Sessao"] = pd.to_datetime(sessoes["Data_Hora_Sessao"])
ingressos["Data_Venda"] = pd.to_datetime(ingressos["Data_Venda"])

# Valores em dinheiro sempre como número decimal
sessoes["Preco_Base"] = sessoes["Preco_Base"].astype(float)
ingressos["Valor_Pago"] = ingressos["Valor_Pago"].astype(float)
bomboniere["Valor_Total"] = bomboniere["Valor_Total"].astype(float)

print(sessoes.dtypes)

ID_Sessao                    int64
ID_Filme                     int64
ID_Sala                      int64
Data_Hora_Sessao    datetime64[ns]
Preco_Base                 float64
dtype: object


In [ ]:
# 5.2 Remove espaços no início e no fim dos textos
colunas_texto = {
    "Dim_Filme": ["Titulo", "Genero", "Classificacao_Indicativa"],
    "Dim_Sala": ["Nome_Sala", "Tipo_Sala"],
    "Fato_Venda_Ingresso": ["Tipo_Ingresso", "Canal_Venda"],
    "Fato_Venda_Bomboniere": ["Item_Comprado"],
}

for nome, colunas in colunas_texto.items():
    for coluna in colunas:
        tabelas[nome][coluna] = tabelas[nome][coluna].str.strip()

print("Textos padronizados.")

Textos padronizados.


In [ ]:
# 5.3 A chave de cada tabela não pode se repetir
chaves = {
    "Dim_Filme": "ID_Filme",
    "Dim_Sala": "ID_Sala",
    "Fato_Sessao": "ID_Sessao",
    "Fato_Venda_Ingresso": "ID_Venda_Ingresso",
    "Fato_Venda_Bomboniere": "ID_Venda_Bomboniere",
}

for nome, df in tabelas.items():
    antes = len(df)
    df.drop_duplicates(subset=chaves[nome], keep="first", inplace=True)
    print(nome, "| linhas removidas:", antes - len(df))

Dim_Filme | linhas removidas: 0
Dim_Sala | linhas removidas: 0
Fato_Sessao | linhas removidas: 0
Fato_Venda_Ingresso | linhas removidas: 0
Fato_Venda_Bomboniere | linhas removidas: 0


In [ ]:
# 5.4 Tabelas auxiliares para validar as regras de negócio

# Quantos ingressos cada sessão vendeu
ingressos_por_sessao = (ingressos.groupby("ID_Sessao").size()
                        .reset_index(name="Ingressos_Vendidos"))

# Cada sessão com a capacidade da sua sala e os ingressos vendidos
lotacao = (sessoes
           .merge(salas[["ID_Sala", "Capacidade"]], on="ID_Sala", how="left")
           .merge(ingressos_por_sessao, on="ID_Sessao", how="left"))

# Cada ingresso com o preço e o horário da sua sessão
vendas = ingressos.merge(sessoes[["ID_Sessao", "Preco_Base", "Data_Hora_Sessao"]],
                         on="ID_Sessao", how="left")

# Valor que cada ingresso deveria custar: meia = metade, inteira = preço cheio
valor_esperado = np.where(vendas["Tipo_Ingresso"] == "Meia",
                          vendas["Preco_Base"] / 2, vendas["Preco_Base"])

# Cada sessão com a data de lançamento do seu filme
estreias = sessoes.merge(filmes[["ID_Filme", "Data_Lancamento"]], on="ID_Filme", how="left")

lotacao.head()

,ID_Sessao,ID_Filme,ID_Sala,Data_Hora_Sessao,Preco_Base,Capacidade,Ingressos_Vendidos
0,1,16,3,2023-11-16 14:00:00,30.0,120,32
1,2,15,15,2023-11-16 14:00:00,70.0,60,10
2,3,12,3,2023-11-16 16:30:00,30.0,120,37
3,4,11,8,2023-11-16 16:30:00,70.0,60,23
4,5,11,13,2023-11-16 16:30:00,45.0,200,75


In [ ]:
# Cada linha é uma regra. O número mostra quantos registros a descumprem.
validacoes = {
    "Sessões com filme que não existe em Dim_Filme":
        (~sessoes["ID_Filme"].isin(filmes["ID_Filme"])).sum(),
    "Sessões com sala que não existe em Dim_Sala":
        (~sessoes["ID_Sala"].isin(salas["ID_Sala"])).sum(),
    "Ingressos de sessão que não existe":
        (~ingressos["ID_Sessao"].isin(sessoes["ID_Sessao"])).sum(),
    "Vendas de bomboniere de sessão que não existe":
        (~bomboniere["ID_Sessao"].isin(sessoes["ID_Sessao"])).sum(),
    "Sessões com mais ingressos que a capacidade da sala":
        (lotacao["Ingressos_Vendidos"] > lotacao["Capacidade"]).sum(),
    "Ingressos com valor diferente da regra inteira/meia":
        (~np.isclose(vendas["Valor_Pago"], valor_esperado)).sum(),
    "Ingressos vendidos depois do início da sessão":
        (vendas["Data_Venda"] >= vendas["Data_Hora_Sessao"]).sum(),
    "Sessões antes da data de lançamento do filme":
        (estreias["Data_Hora_Sessao"] < estreias["Data_Lancamento"]).sum(),
    "Duas sessões na mesma sala e no mesmo horário":
        sessoes.duplicated(subset=["ID_Sala", "Data_Hora_Sessao"]).sum(),
    "Vendas de bomboniere com quantidade ou valor menor ou igual a zero":
        ((bomboniere["Quantidade"] <= 0) | (bomboniere["Valor_Total"] <= 0)).sum(),
}

resultado = pd.DataFrame({
    "Regra": list(validacoes),
    "Registros com problema": [int(v) for v in validacoes.values()],
})
display(resultado)

,Regra,Registros com problema
0,Sessões com filme que não existe em Dim_Filme,0
1,Sessões com sala que não existe em Dim_Sala,0
2,Ingressos de sessão que não existe,0
3,Vendas de bomboniere de sessão que não existe,0
4,Sessões com mais ingressos que a capacidade da...,0
5,Ingressos com valor diferente da regra inteira...,0
6,Ingressos vendidos depois do início da sessão,0
7,Sessões antes da data de lançamento do filme,0
8,Duas sessões na mesma sala e no mesmo horário,0
9,Vendas de bomboniere com quantidade ou valor m...,0


In [ ]:
# Primeiras linhas das tabelas com datas
display(sessoes.head(3))
display(ingressos.head(3))

,ID_Sessao,ID_Filme,ID_Sala,Data_Hora_Sessao,Preco_Base
0,1,16,3,2023-11-16 14:00:00,30.0
1,2,15,15,2023-11-16 14:00:00,70.0
2,3,12,3,2023-11-16 16:30:00,30.0


,ID_Venda_Ingresso,ID_Sessao,Tipo_Ingresso,Canal_Venda,Data_Venda,Valor_Pago
0,1,5,Meia,Site,2023-11-11 08:18:00,22.5
1,2,7,Inteira,Site,2023-11-11 08:33:00,45.0
2,3,3,Meia,Site,2023-11-11 08:50:00,15.0


In [ ]:
## 6. Enriquecimento: novas colunas para a análise

In [ ]:
# 6.1 Sessões: data, horário e tempo em cartaz

# Separa a data e o horário da sessão
sessoes["Data"] = sessoes["Data_Hora_Sessao"].dt.normalize()   # zera o horário
sessoes["Hora"] = sessoes["Data_Hora_Sessao"].dt.strftime("%H:%M")

# np.where funciona como um SE: condição, valor se verdadeiro, valor se falso
sessoes["Faixa_Horaria"] = np.where(sessoes["Data_Hora_Sessao"].dt.hour < 18, "Tarde", "Noite")

# Busca a data de lançamento de cada filme usando o ID_Filme
data_lancamento = filmes.set_index("ID_Filme")["Data_Lancamento"]
sessoes["Dias_Em_Cartaz"] = (sessoes["Data"] - sessoes["ID_Filme"].map(data_lancamento)).dt.days

# pd.cut divide um número em faixas. right=False: o limite da direita fica de fora.
sessoes["Fase_Lancamento"] = pd.cut(
    sessoes["Dias_Em_Cartaz"],
    bins=[0, 31, 91, np.inf],
    labels=["1 - Estreia (até 30 dias)", "2 - De 31 a 90 dias", "3 - Mais de 90 dias"],
    right=False,
).astype(str)

sessoes.head()

,ID_Sessao,ID_Filme,ID_Sala,Data_Hora_Sessao,Preco_Base,Data,Hora,Faixa_Horaria,Dias_Em_Cartaz,Fase_Lancamento
0,1,16,3,2023-11-16 14:00:00,30.0,2023-11-16,14:00,Tarde,15,1 - Estreia (até 30 dias)
1,2,15,15,2023-11-16 14:00:00,70.0,2023-11-16,14:00,Tarde,86,2 - De 31 a 90 dias
2,3,12,3,2023-11-16 16:30:00,30.0,2023-11-16,16:30,Tarde,35,2 - De 31 a 90 dias
3,4,11,8,2023-11-16 16:30:00,70.0,2023-11-16,16:30,Tarde,11,1 - Estreia (até 30 dias)
4,5,11,13,2023-11-16 16:30:00,45.0,2023-11-16,16:30,Tarde,11,1 - Estreia (até 30 dias)


In [ ]:
# 6.2 Tabela calendário: uma linha por dia

NOMES_DIA = {0: "Segunda", 1: "Terça", 2: "Quarta", 3: "Quinta",
             4: "Sexta", 5: "Sábado", 6: "Domingo"}
NOMES_MES = {1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril", 5: "Maio", 6: "Junho",
             7: "Julho", 8: "Agosto", 9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"}

# Todos os dias entre a primeira e a última sessão
calendario = pd.DataFrame({
    "Data": pd.date_range(sessoes["Data"].min(), sessoes["Data"].max(), freq="D")
})

calendario["Ano"] = calendario["Data"].dt.year
calendario["Num_Mes"] = calendario["Data"].dt.month
calendario["Mes"] = calendario["Num_Mes"].map(NOMES_MES)
calendario["Dia"] = calendario["Data"].dt.day
calendario["Num_Dia_Semana"] = calendario["Data"].dt.dayofweek + 1
calendario["Dia_Semana"] = calendario["Data"].dt.dayofweek.map(NOMES_DIA)
calendario["Tipo_Dia"] = np.where(calendario["Num_Dia_Semana"] >= 6, "Fim de semana", "Dia útil")

# Semana de exibição: conta de 7 em 7 dias a partir do primeiro dia
calendario["Semana"] = (calendario["Data"] - calendario["Data"].min()).dt.days // 7 + 1

print("Calendário com", len(calendario), "dias")
calendario.head(8)

Calendário com 28 dias


,Data,Ano,Num_Mes,Mes,Dia,Num_Dia_Semana,Dia_Semana,Tipo_Dia,Semana
0,2023-11-16,2023,11,Novembro,16,4,Quinta,Dia útil,1
1,2023-11-17,2023,11,Novembro,17,5,Sexta,Dia útil,1
2,2023-11-18,2023,11,Novembro,18,6,Sábado,Fim de semana,1
3,2023-11-19,2023,11,Novembro,19,7,Domingo,Fim de semana,1
4,2023-11-20,2023,11,Novembro,20,1,Segunda,Dia útil,1
5,2023-11-21,2023,11,Novembro,21,2,Terça,Dia útil,1
6,2023-11-22,2023,11,Novembro,22,3,Quarta,Dia útil,1
7,2023-11-23,2023,11,Novembro,23,4,Quinta,Dia útil,2


In [ ]:
# 6.3 Ingressos: antecedência da compra e tipo de canal

# Horário de início de cada sessão, buscado pelo ID_Sessao
inicio_sessao = sessoes.set_index("ID_Sessao")["Data_Hora_Sessao"]
diferenca = ingressos["ID_Sessao"].map(inicio_sessao) - ingressos["Data_Venda"]

# total_seconds() devolve a diferença em segundos; dividir por 3600 converte para horas
ingressos["Antecedencia_Horas"] = (diferenca.dt.total_seconds() / 3600).round(1)

ingressos["Faixa_Antecedencia"] = pd.cut(
    ingressos["Antecedencia_Horas"],
    bins=[0, 3, 24, 72, np.inf],
    labels=["1 - Até 3 horas", "2 - De 3 a 24 horas", "3 - De 1 a 3 dias", "4 - Mais de 3 dias"],
    right=False,
).astype(str)

# isin verifica se o valor está na lista
ingressos["Tipo_Canal"] = np.where(ingressos["Canal_Venda"].isin(["Site", "Aplicativo"]),
                                   "Digital", "Presencial")

ingressos.head()

,ID_Venda_Ingresso,ID_Sessao,Tipo_Ingresso,Canal_Venda,Data_Venda,Valor_Pago,Antecedencia_Horas,Faixa_Antecedencia,Tipo_Canal
0,1,5,Meia,Site,2023-11-11 08:18:00,22.5,128.2,4 - Mais de 3 dias,Digital
1,2,7,Inteira,Site,2023-11-11 08:33:00,45.0,130.4,4 - Mais de 3 dias,Digital
2,3,3,Meia,Site,2023-11-11 08:50:00,15.0,127.7,4 - Mais de 3 dias,Digital
3,4,5,Meia,Aplicativo,2023-11-11 11:07:00,22.5,125.4,4 - Mais de 3 dias,Digital
4,5,1,Meia,Site,2023-11-11 11:29:00,15.0,122.5,4 - Mais de 3 dias,Digital


In [ ]:
# 6.4 Bomboniere: preço unitário e categoria do item

def classificar_item(item):
    """Devolve a categoria de um item da bomboniere."""
    if item.startswith("Combo"):
        return "Combo"
    if "Pipoca" in item:
        return "Pipoca"
    if any(palavra in item for palavra in ("Refrigerante", "Água", "Energético")):
        return "Bebida"
    return "Doces e lanches"


bomboniere["Preco_Unitario"] = bomboniere["Valor_Total"] / bomboniere["Quantidade"]
# apply executa a função em cada linha da coluna
bomboniere["Categoria_Item"] = bomboniere["Item_Comprado"].apply(classificar_item)

# Conferência: cada item em uma única categoria, com um único preço
(bomboniere.groupby(["Categoria_Item", "Item_Comprado"])
 .agg(Preco_Unitario=("Preco_Unitario", "first"), Vendas=("Quantidade", "count"))
 .reset_index())

,Categoria_Item,Item_Comprado,Preco_Unitario,Vendas
0,Bebida,Energético,18.0,283
1,Bebida,Refrigerante Grande,15.0,468
2,Bebida,Refrigerante Médio,12.0,620
3,Bebida,Água Mineral,5.0,499
4,Combo,Combo Individual (Pipoca M + 1 Refri),40.0,972
5,Combo,Combo Kids,35.0,351
6,Combo,Combo Mega (Pipoca G + 2 Refri),65.0,767
7,Combo,Combo VIP (Pipoca Trufada + Vinho),95.0,47
8,Doces e lanches,Chocolate Barra,8.0,454
9,Doces e lanches,Nachos com Queijo,32.0,378


In [ ]:
## 7. Análise exploratória: prévia dos insights

In [ ]:
# 7.0 Tabela resumo: uma linha por sessão, juntando as 5 tabelas

# Total de ingressos e de receita por sessão
ingressos_sessao = (ingressos.groupby("ID_Sessao")
                    .agg(Ingressos_Vendidos=("ID_Venda_Ingresso", "count"),
                         Receita_Ingressos=("Valor_Pago", "sum"))
                    .reset_index())

bomboniere_sessao = (bomboniere.groupby("ID_Sessao")
                     .agg(Receita_Bomboniere=("Valor_Total", "sum"))
                     .reset_index())

resumo = (sessoes
          .merge(filmes[["ID_Filme", "Titulo", "Genero", "Classificacao_Indicativa"]],
                 on="ID_Filme", how="left")
          .merge(salas, on="ID_Sala", how="left")
          .merge(calendario[["Data", "Num_Dia_Semana", "Dia_Semana", "Tipo_Dia", "Semana"]],
                 on="Data", how="left")
          .merge(ingressos_sessao, on="ID_Sessao", how="left")
          .merge(bomboniere_sessao, on="ID_Sessao", how="left"))

# Sessão sem venda fica com valor nulo depois do merge: trocamos por zero
colunas_vendas = ["Ingressos_Vendidos", "Receita_Ingressos", "Receita_Bomboniere"]
resumo[colunas_vendas] = resumo[colunas_vendas].fillna(0)

# O merge não pode criar nem perder sessões
assert len(resumo) == len(sessoes), "O número de sessões mudou depois do merge"
print("Tabela resumo:", len(resumo), "sessões")

Tabela resumo: 336 sessões


In [ ]:
# 7.1 Indicadores gerais do cinema
total_sessoes = len(resumo)
total_ingressos = int(resumo["Ingressos_Vendidos"].sum())
lugares_ofertados = int(resumo["Capacidade"].sum())
receita_ingressos = resumo["Receita_Ingressos"].sum()
receita_bomboniere = resumo["Receita_Bomboniere"].sum()
receita_total = receita_ingressos + receita_bomboniere

print(f"Sessões realizadas:          {total_sessoes}")
print(f"Lugares ofertados:           {lugares_ofertados:,}")
print(f"Ingressos vendidos:          {total_ingressos:,}")
print(f"Taxa de ocupação:            {total_ingressos / lugares_ofertados:.1%}")
print(f"Receita de ingressos:        R$ {receita_ingressos:,.2f}")
print(f"Receita de bomboniere:       R$ {receita_bomboniere:,.2f}")
print(f"Receita total:               R$ {receita_total:,.2f}")
print(f"Participação da bomboniere:  {receita_bomboniere / receita_total:.1%}")
print(f"Ticket médio do ingresso:    R$ {receita_ingressos / total_ingressos:,.2f}")
print(f"Bomboniere por ingresso:     R$ {receita_bomboniere / total_ingressos:,.2f}")

Sessões realizadas:          336
Lugares ofertados:           50,940
Ingressos vendidos:          22,235
Taxa de ocupação:            43.6%
Receita de ingressos:        R$ 742,397.50
Receita de bomboniere:       R$ 285,864.00
Receita total:               R$ 1,028,261.50
Participação da bomboniere:  27.8%
Ticket médio do ingresso:    R$ 33.39
Bomboniere por ingresso:     R$ 12.86


In [ ]:
# 7.2 Ocupação por agrupamento

def ocupacao_por(colunas):
    """Resume sessões, lugares, ingressos, receita e ocupação por agrupamento."""
    tabela = (resumo.groupby(colunas)
              .agg(Sessoes=("ID_Sessao", "count"),
                   Lugares=("Capacidade", "sum"),
                   Ingressos=("Ingressos_Vendidos", "sum"),
                   Receita_Ingressos=("Receita_Ingressos", "sum"))
              .reset_index())
    tabela["Ocupacao_%"] = (100 * tabela["Ingressos"] / tabela["Lugares"]).round(1)
    return tabela


print("Por dia da semana")
display(ocupacao_por(["Num_Dia_Semana", "Dia_Semana"]))

print("Por horário")
display(ocupacao_por("Hora"))

print("Por tipo de sala")
display(ocupacao_por("Tipo_Sala"))

print("Por gênero")
display(ocupacao_por("Genero").sort_values("Ocupacao_%", ascending=False))

Por dia da semana


,Num_Dia_Semana,Dia_Semana,Sessoes,Lugares,Ingressos,Receita_Ingressos,Ocupacao_%
0,1,Segunda,32,4650,1103,36290.0,23.7
1,2,Terça,32,4600,1430,48260.0,31.1
2,3,Quarta,40,6050,1831,63077.5,30.3
3,4,Quinta,40,6380,2154,68182.5,33.8
4,5,Sexta,56,8830,4049,135155.0,45.9
5,6,Sábado,72,10460,6300,212492.5,60.2
6,7,Domingo,64,9970,5368,178940.0,53.8


Por horário


,Hora,Sessoes,Lugares,Ingressos,Receita_Ingressos,Ocupacao_%
0,14:00,70,10650,2707,88607.5,25.4
1,16:30,83,13350,4831,159152.5,36.2
2,19:00,98,14730,8216,272225.0,55.8
3,21:30,85,12210,6481,222412.5,53.1


Por tipo de sala


,Tipo_Sala,Sessoes,Lugares,Ingressos,Receita_Ingressos,Ocupacao_%
0,3D XPLUS,85,17000,7078,245407.5,41.6
1,IMAX,43,10750,4258,164625.0,39.6
2,Padrão,135,18810,8687,201885.0,46.2
3,VIP,73,4380,2212,130480.0,50.5


Por gênero


,Genero,Sessoes,Lugares,Ingressos,Receita_Ingressos,Ocupacao_%
7,Terror,13,2030,1171,41575.0,57.7
1,Ação,78,12200,6324,220972.5,51.8
0,Aventura,74,12010,5096,159312.5,42.4
2,Comédia,52,7040,2963,98547.5,42.1
4,Ficção Científica,51,8080,3404,115595.0,42.1
5,Romance,29,3730,1395,45280.0,37.4
3,Drama,32,4770,1627,52817.5,34.1
6,Suspense,7,1080,255,8297.5,23.6


In [ ]:
# 7.3 Canais de venda
canais = (ingressos.groupby(["Tipo_Canal", "Canal_Venda"])
          .agg(Ingressos=("ID_Venda_Ingresso", "count"),
               Receita=("Valor_Pago", "sum"),
               Ticket_Medio=("Valor_Pago", "mean"),
               Antecedencia_Media_Horas=("Antecedencia_Horas", "mean"))
          .reset_index())
canais["Participacao_%"] = (100 * canais["Ingressos"] / canais["Ingressos"].sum()).round(1)
display(canais)

# Meia-entrada por tipo de sala
tipo_sala_por_sessao = resumo.set_index("ID_Sessao")["Tipo_Sala"]
meia = (ingressos.assign(Tipo_Sala=ingressos["ID_Sessao"].map(tipo_sala_por_sessao),
                         Meia=(ingressos["Tipo_Ingresso"] == "Meia"))
        .groupby("Tipo_Sala")
        .agg(Ingressos=("Meia", "count"), Meias=("Meia", "sum"))
        .reset_index())
meia["Meia_%"] = (100 * meia["Meias"] / meia["Ingressos"]).round(1)
display(meia)

,Tipo_Canal,Canal_Venda,Ingressos,Receita,Ticket_Medio,Antecedencia_Media_Horas,Participacao_%
0,Digital,Aplicativo,6158,207700.0,33.728483,32.090289,27.7
1,Digital,Site,8667,292990.0,33.805238,32.316361,39.0
2,Presencial,Bilheteria,4680,153012.5,32.694979,1.535321,21.0
3,Presencial,Totem,2730,88695.0,32.489011,1.572601,12.3


,Tipo_Sala,Ingressos,Meias,Meia_%
0,3D XPLUS,7078,3249,45.9
1,IMAX,4258,1931,45.3
2,Padrão,8687,3915,45.1
3,VIP,2212,696,31.5


In [ ]:
# 7.4 Bomboniere por categoria
categorias = (bomboniere.groupby("Categoria_Item")
              .agg(Itens_Vendidos=("Quantidade", "sum"), Receita=("Valor_Total", "sum"))
              .reset_index()
              .sort_values("Receita", ascending=False))
categorias["Participacao_%"] = (100 * categorias["Receita"] / categorias["Receita"].sum()).round(1)
display(categorias)

,Categoria_Item,Itens_Vendidos,Receita,Participacao_%
1,Combo,2776,137135.0,48.0
3,Pipoca,3034,89211.0,31.2
2,Doces e lanches,1443,30872.0,10.8
0,Bebida,2444,28646.0,10.0


In [ ]:
# 8.1 Grava os 6 arquivos tratados em uma subpasta do Drive
PASTA_TRATADOS = PASTA + "dados_tratados/"
os.makedirs(PASTA_TRATADOS, exist_ok=True)

# Cópias para a exportação: as datas sem horário viram texto AAAA-MM-DD
filmes_final = filmes.copy()
filmes_final["Data_Lancamento"] = filmes_final["Data_Lancamento"].dt.strftime("%Y-%m-%d")

sessoes_final = sessoes.copy()
sessoes_final["Data"] = sessoes_final["Data"].dt.strftime("%Y-%m-%d")

calendario_final = calendario.copy()
calendario_final["Data"] = calendario_final["Data"].dt.strftime("%Y-%m-%d")

saidas = {
    "Dim_Filme": filmes_final,
    "Dim_Sala": salas,
    "Dim_Calendario": calendario_final,
    "Fato_Sessao": sessoes_final,
    "Fato_Venda_Ingresso": ingressos,
    "Fato_Venda_Bomboniere": bomboniere,
}

for nome, df in saidas.items():
    # index=False: não grava a numeração do pandas como coluna extra
    df.to_csv(PASTA_TRATADOS + nome + ".csv", index=False, encoding="utf-8")
    print(nome + ".csv:", len(df), "linhas,", df.shape[1], "colunas")

Dim_Filme.csv: 20 linhas, 5 colunas
Dim_Sala.csv: 20 linhas, 4 colunas
Dim_Calendario.csv: 28 linhas, 9 colunas
Fato_Sessao.csv: 336 linhas, 10 colunas
Fato_Venda_Ingresso.csv: 22235 linhas, 9 colunas
Fato_Venda_Bomboniere.csv: 7454 linhas, 7 colunas
